In [1]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

import optuna

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


# Init

## Init Reservoir

In [2]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [3]:
u_val = 1
u_dist = 400
u_dataset = u_val * np.sin(2 * np.pi * t / (2 * u_dist))
u_dataset = u_dataset[transient_steps_chaos:]

## Init Funcs

In [4]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [5]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [6]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [7]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    combined_weights = weights.reshape(2, -1).T.flatten()

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [8]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [18]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [19]:
@njit(cache=True)
def run_simulation(
    steps,
    dt,
    m_inv_diag,
    c_diag,
    U,
    initial_pos,
    connections_list,
    k_vals,
    rest_lens,
    wall_nodes=[-1],
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + 0.5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Non Lin

In [20]:
x = np.array([0, 0, 0])
y = np.array([0, 1, 2])

nodes_pos = np.column_stack((x, y))

In [37]:
tau_steps = 1
free_steps = 3
dt = 0.01
input_force = 0.4
m_val = 2
c_val = 0.3
k_val = 10

target_nodes = np.array([1])
wall_nodes = [0, 2]

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_nodes = np.ones(num_nodes) * m_val
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / m_diag

c_nodes = np.ones(num_nodes) * c_val
c_diag = np.repeat(c_nodes, dims)

In [52]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, -1)

k_vals = np.ones(src_nodes.shape[0]) * k_val
connections_list = np.column_stack((src_nodes, dst_nodes))
rest_lens = np.ones(src_nodes.shape[0]) * 1.1

In [53]:
force_data = u_dataset.reshape(-1, 1)
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
col_indices = col_indices[:1]
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]
U = U * input_force

In [54]:
displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U,
    rest_lens=rest_lens,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=wall_nodes,
)

In [55]:
X = np.column_stack((displacement, velocity))
X_delayed = X[: -tau_steps * free_steps]
X_data = X_delayed[transient_steps_reservoir * free_steps :]
Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)

rest_test_steps = test_steps * free_steps
X_train, X_test = (
    X_data[:-rest_test_steps],
    X_data[-rest_test_steps:],
)
Y_train, Y_test = (
    Y_data[:-rest_test_steps],
    Y_data[-rest_test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9952 0.0489


In [56]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    wall_nodes=wall_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [57]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Lin

In [62]:
N = 2

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [74]:
tau_steps = 1
free_steps = 3
dt = 0.01
input_force = 0.4*10
m_val = 2
c_val = 0.3
k_val = 10

target_nodes = np.array([-1])
wall_nodes = [0]

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_nodes = np.ones(num_nodes) * m_val
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / m_diag

c_nodes = np.ones(num_nodes) * c_val
c_diag = np.repeat(c_nodes, dims)

In [75]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, -1)

k_vals = np.ones(src_nodes.shape[0]) * k_val
connections_list = np.column_stack((src_nodes, dst_nodes))
rest_lens = np.ones(src_nodes.shape[0])

In [76]:
force_data = u_dataset.reshape(-1, 1)
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
col_indices = col_indices[:1]
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]
U = U * input_force

In [77]:
displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U,
    initial_pos=nodes_pos,
    rest_lens=rest_lens,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=wall_nodes,
)

In [80]:
X = np.column_stack((displacement, velocity))
X_delayed = X[: -tau_steps * free_steps]
X_data = X_delayed[transient_steps_reservoir * free_steps :]
Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)

rest_test_steps = test_steps * free_steps
X_train, X_test = (
    X_data[:-rest_test_steps],
    X_data[-rest_test_steps:],
)
Y_train, Y_test = (
    Y_data[:-rest_test_steps],
    Y_data[-rest_test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")
print(r_2, mse)

1.0000 0.0020
0.9999923790119248 0.001952048677058638


In [79]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    wall_nodes=wall_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [73]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Non Lin Bayesian

In [81]:
x = np.array([0, 0, 0])
y = np.array([0, 1, 2])

nodes_pos = np.column_stack((x, y))

In [82]:
tau_steps = 1
free_steps = 3
dt = 0.01

target_nodes = np.array([1])
wall_nodes = [0, 2]

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [84]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, -1)

k_vals = np.ones(src_nodes.shape[0]) * k_val
connections_list = np.column_stack((src_nodes, dst_nodes))
rest_lens = np.ones(src_nodes.shape[0]) * 1.1

In [85]:
force_data = u_dataset.reshape(-1, 1)
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
col_indices = col_indices[:1]
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

In [87]:
def bayesian_trial(m_val, c_val, k_val, input_force, ridge_alpha):
    m_nodes = np.ones(num_nodes) * m_val
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = np.ones(num_nodes) * c_val
    c_diag = np.repeat(c_nodes, dims)

    k_vals = np.ones(src_nodes.shape[0]) * k_val

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        rest_lens=rest_lens,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=wall_nodes,
    )

    X = np.column_stack((displacement, velocity))

    positions = nodes_pos + displacement.reshape(-1, num_nodes, dims)
    pos_a = positions[:, connections_list[:, 0], :]
    pos_b = positions[:, connections_list[:, 1], :]
    distances = np.linalg.norm(pos_a - pos_b, axis=2)
    if np.any(distances < 0.05):
        return (), (-1.0, 1e9), ()

    if np.any(np.isnan(X)):
        return (), (-1.0, 1e9), ()

    X_delayed = X[: -tau_steps * free_steps]
    X_data = X_delayed[transient_steps_reservoir * free_steps :]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)

    rest_test_steps = test_steps * free_steps
    X_train, X_test = (
        X_data[:-rest_test_steps],
        X_data[-rest_test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-rest_test_steps],
        Y_data[-rest_test_steps:],
    )

    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity)

In [88]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    2, 0.3, 10, 0.4, 0.1
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9952 0.0489


In [89]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.01, 10, log=True),
        trial.suggest_float("damping", 0.01, 10, log=True),
        trial.suggest_float("stiffness", 0.01, 10, log=True),
        trial.suggest_float("input_force", 1e-3, 100.0, log=True),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
    )

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[I 2026-07-23 09:37:38,806] A new study created in memory with name: no-name-8452bf7e-c652-4dc5-afec-3d6451a3f43b


[Optuna] Processing Trial #99...

In [90]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #71
  Values: [0.9999908260495461, 0.0021417224906414307]
  Params: {'mass': 0.4539874716435445, 'damping': 0.07636158443771728, 'stiffness': 0.05313075111585172, 'input_force': 39.47822479824915, 'ridge_alpha': 0.022992318561584752}


In [91]:
params = study.best_trials[0].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness"],
    params["input_force"],
    params["ridge_alpha"],
)

print(r_2, mse)

0.9999908260495461 0.0021417224906414307


In [92]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
    vel=velocity,
).show()

In [182]:
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Lin Bayesian

In [105]:
N = 2

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [106]:
tau_steps = 1
free_steps = 3
dt = 0.01

target_nodes = np.array([-1])
wall_nodes = [0]

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [107]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, -1)

k_vals = np.ones(src_nodes.shape[0]) * k_val
connections_list = np.column_stack((src_nodes, dst_nodes))
rest_lens = np.ones(src_nodes.shape[0])

In [108]:
force_data = u_dataset.reshape(-1, 1)
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
col_indices = col_indices[:1]
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

In [109]:
def bayesian_trial(m_val, c_val, k_val, input_force, ridge_alpha):
    m_nodes = np.ones(num_nodes) * m_val
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = np.ones(num_nodes) * c_val
    c_diag = np.repeat(c_nodes, dims)

    k_vals = np.ones(src_nodes.shape[0]) * k_val

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        rest_lens=rest_lens,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=wall_nodes,
    )

    X = np.column_stack((displacement, velocity))

    positions = nodes_pos + displacement.reshape(-1, num_nodes, dims)
    pos_a = positions[:, connections_list[:, 0], :]
    pos_b = positions[:, connections_list[:, 1], :]
    distances = np.linalg.norm(pos_a - pos_b, axis=2)
    if np.any(distances < 0.05):
        return (), (-1.0, 1e9), ()

    if np.any(np.isnan(X)):
        return (), (-1.0, 1e9), ()

    X_delayed = X[: -tau_steps * free_steps]
    X_data = X_delayed[transient_steps_reservoir * free_steps :]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)

    rest_test_steps = test_steps * free_steps
    X_train, X_test = (
        X_data[:-rest_test_steps],
        X_data[-rest_test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-rest_test_steps],
        Y_data[-rest_test_steps:],
    )

    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity)

In [110]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    2, 0.3, 10, 0.4, 0.1
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9928 0.0600


In [111]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.01, 10, log=True),
        trial.suggest_float("damping", 0.01, 10, log=True),
        trial.suggest_float("stiffness", 0.01, 10, log=True),
        trial.suggest_float("input_force", 1e-3, 100.0, log=True),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
    )

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #44...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real


[Optuna] Processing Trial #99...

In [ ]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #17
  Values: [0.999994603523026, 0.001642631573715739]
  Params: {'mass': 3.2128027302344617, 'damping': 0.4699272100917341, 'stiffness': 5.996819286212312, 'input_force': 2.8973844506960265, 'ridge_alpha': 0.008185421633563705}


: 

In [102]:
params = study.best_trials[0].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness"],
    params["input_force"],
    params["ridge_alpha"],
)

print(r_2, mse)

0.9999953181365573 0.0015300103664205346


In [103]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [ ]:
# weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Non Lin Bayesian Multiple

In [12]:
N = 3
dist_between = 2.5

x = np.zeros(N*3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 1, 0]) + dist_between*i
y = np.tile(np.array([0, 1, 2]), N)

nodes_pos = np.column_stack((x, y))

In [13]:
nodes_pos

array([[0. , 0. ],
       [1. , 1. ],
       [0. , 2. ],
       [2.5, 0. ],
       [3.5, 1. ],
       [2.5, 2. ],
       [5. , 0. ],
       [6. , 1. ],
       [5. , 2. ]])

In [14]:
plot_grid(nodes_pos).show()

In [15]:
tau_steps = 1
free_steps = 3
dt = 0.01

target_nodes = np.array([1])
starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [16]:
node_ids = np.arange(x.size)

# 0->1 1->2 then 3->4 4->5 then 6->7 7->8
# 0, 1, 3, 4, 6, 7
# 1, 2, 4, 5, 7, 8
starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()

# Between Connections
# 1->4 then 4->7
between_src = np.array([1, 4])
between_dst = np.array([4, 7])

src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])

connections_list = np.column_stack((src_nodes, dst_nodes))

In [17]:
force_data = u_dataset.reshape(-1, 1)
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
col_indices = col_indices[:1]
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

In [18]:
def bayesian_trial(m_val, c_val, k_val, k_between_val, input_force, ridge_alpha):
    m_nodes = np.ones(num_nodes) * m_val
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = np.ones(num_nodes) * c_val
    c_diag = np.repeat(c_nodes, dims)

    k_connection_vals = np.ones(connection_src.shape[0]) * k_val
    k_between_vals = np.ones(between_src.shape[0]) * k_between_val
    k_vals = np.concatenate([k_connection_vals, k_between_vals])

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=wall_nodes,
    )

    X = np.column_stack((displacement, velocity))

    positions = nodes_pos + displacement.reshape(-1, num_nodes, dims)
    pos_a = positions[:, connections_list[:, 0], :]
    pos_b = positions[:, connections_list[:, 1], :]
    distances = np.linalg.norm(pos_a - pos_b, axis=2)
    if np.any(distances < 0.05):
        return (), (-1.0, 1e9), ()

    if np.any(np.isnan(X)):
        return (), (-1.0, 1e9), ()

    X_delayed = X[: -tau_steps * free_steps]
    X_data = X_delayed[transient_steps_reservoir * free_steps :]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)

    rest_test_steps = test_steps * free_steps
    X_train, X_test = (
        X_data[:-rest_test_steps],
        X_data[-rest_test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-rest_test_steps],
        Y_data[-rest_test_steps:],
    )

    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity)

In [19]:
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    2, 0.3, 10, 10, 0.4, 0.1
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9978 0.0332


In [20]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.01, 10),
        trial.suggest_float("damping", 0.01, 10),
        trial.suggest_float("stiffness_conn", 0.01, 10),
        trial.suggest_float("stiffness_between", 0.01, 10),
        trial.suggest_float("input_force", 1e-3, 100.0),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
    )

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, timeout=1200, n_jobs=-1)

[I 2026-07-20 13:12:45,056] A new study created in memory with name: no-name-f9929d0e-8561-4929-b75a-6d8a5ec9a083


[Optuna] Processing Trial #13478...

In [21]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #9156
  Values: [0.9999954297189831, 0.0015116681211314066]
  Params: {'mass': 6.599782772904393, 'damping': 1.6536187909308453, 'stiffness_conn': 0.2770684899239264, 'stiffness_between': 1.4075817426887467, 'input_force': 5.8117604582398705, 'ridge_alpha': 0.0010068488564346227}
Trial #9217
  Values: [0.9999954297189831, 0.0015116681211314066]
  Params: {'mass': 6.599782772904393, 'damping': 1.6536187909308453, 'stiffness_conn': 0.2770684899239264, 'stiffness_between': 1.4075817426887467, 'input_force': 5.8117604582398705, 'ridge_alpha': 0.0010068488564346227}
Trial #9240
  Values: [0.9999954297189831, 0.0015116681211314066]
  Params: {'mass': 6.599782772904393, 'damping': 1.6536187909308453, 'stiffness_conn': 0.2770684899239264, 'stiffness_between': 1.4075817426887467, 'input_force': 5.8117604582398705, 'ridge_alpha': 0.0010068488564346227}
Trial #9298
  Values: [0.9999954297189831, 0.0015116681211314066]
  Params: {'mass': 6.599782772904393, 'damping': 1.6536187909308453, 'sti

In [22]:
params = study.best_trials[0].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness_conn"],
    params["stiffness_between"],
    params["input_force"],
    params["ridge_alpha"],
)

print(r_2, mse)

0.9999954297189831 0.0015116681211314066


In [23]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
    vel=velocity,
).show()

2026-07-20 13:39:08.379 python[84516:7064799] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-20 13:39:08.379 python[84516:7064799] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [24]:
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()